***

# **BLS Queries**

***

In this file, we want to try and pull the total number of employment from BLS for different industries. To give some context, the API works by specifying a specific series id, which can be used to pull data from a specific table from BLS. It can also be used to select information from a table, parameters like specific counties or specific industry can be referenced. The survey we are trying to pull from is State and Employment, Hours, and Earnings; which can be found in the following link: https://www.bls.gov/help/hlpforma.htm#EW. 

In [38]:
# Packages

import pandas as pd
import json
import requests

In [2]:
# Base URL for API V2
url = 'https://api.bls.gov/publicAPI/v2/timeseries/data/'

# API key in config.py which contains: bls_key = 'key'
key = '03688c33d1194462aa72e4e97253b1e7'
key = '?registrationkey={}'.format(key)

# Series stored as a dictionary
series_dict = {
    'LNS14000003': 'White',
    'LNS14000006': 'Black',
    'LNS14000009': 'Hispanic'}

# Start year and end year
dates = ('2008', '2017')

In [27]:
# Specify json as content type to return
headers = {'Content-type': 'application/json'}

# Submit the list of series as data
data = json.dumps({
    "seriesid": list(series_dict.keys()),
    "startyear": dates[0],
    "endyear": dates[1]})

# Post request for the data
p = requests.post(
    '{}{}'.format(url, key),
    headers=headers,
    data=data).json()['Results']['series']

In [28]:
# Date index from first series
date_list = [f"{i['year']}-{i['period'][1:]}-01" for i in p[0]['data']]

# Empty dataframe to fill with values
df = pd.DataFrame()

# Build a pandas series from the API results, p
for s in p:
    df[series_dict[s['seriesID']]] = pd.Series(
        index = pd.to_datetime(date_list),
        data = [i['value'] for i in s['data']]
        ).astype(float).iloc[::-1]

# Show last 5 results
df.tail()

,Agricultural Total Employment
2023-11-01,161866.0
2023-12-01,161183.0
2024-01-01,161152.0
2024-02-01,160968.0
2024-03-01,161466.0


In [26]:
# Series stored as a dictionary
series_dict = {
    'LNS12000000': 'Agricultural Total Employment'} # testing a different series ID

# Start year and end year
dates = ('2022', '2024')

# Specify json as content type to return
headers = {'Content-type': 'application/json'}

# Submit the list of series as data
data = json.dumps({
    "seriesid": list(series_dict.keys()),
    "startyear": dates[0],
    "endyear": dates[1]})

# Post request for the data
p = requests.post(
    '{}{}'.format(url, key),
    headers=headers,
    data=data).json()['Results']['series']
# Date index from first series
date_list = [f"{i['year']}-{i['period'][1:]}-01" for i in p[0]['data']]

# Empty dataframe to fill with values
df = pd.DataFrame()

# Build a pandas series from the API results, p
for s in p:
    df[series_dict[s['seriesID']]] = pd.Series(
        index = pd.to_datetime(date_list),
        data = [i['value'] for i in s['data']]
        ).astype(float).iloc[::-1]

# Show last 5 results
df.tail()

,Agricultural Total Employment
2023-11-01,161866.0
2023-12-01,161183.0
2024-01-01,161152.0
2024-02-01,160968.0
2024-03-01,161466.0


***

# **Running Pulls on our Counties Using MSA and Construction as Industry**

***

In [164]:
def bls_query(series_dict, dates, api_key):

    url = 'https://api.bls.gov/publicAPI/v2/timeseries/data/'

    key = '?registrationkey={}'.format(api_key)

    # Specify json as content type to return
    headers = {'Content-type': 'application/json'}

    # Submit the list of series as data
    data = json.dumps({
        "seriesid": list(series_dict.keys()),
        "startyear": dates[0],
        "endyear": dates[1]})

    # Post request for the data
    p = requests.post(
        '{}{}'.format(url, key),
        headers=headers,
        data=data).json()['Results']['series']
    
    # Date index from first series
    date_list = [f"{i['year']}-{i['period'][1:]}-01" for i in p[0]['data']]

    global df

    # Empty dataframe to fill with values
    df = pd.DataFrame()

    # Build a pandas series from the API results, p
    for s in p:
        df[series_dict[s['seriesID']]] = pd.Series(
            index = pd.to_datetime(date_list),
            data = [i['value'] for i in s['data']]
            ).astype(float).iloc[::-1]

    return(df)

In [20]:
# Making Series ID
series_list = []
front = 'SMU06'

industry = '20236000'

type = '01'

# Making a county list for multiple series id creation | In order: Sac + Placer + El Dorado Yolo, Yuba + Sutter,
county_list = ['40900', '49700']
for i in county_list:
    id = front + i + tail + industry + type
    series_list.append(id)

In [21]:
series_list

['SMU064090050012023600001', 'SMU064970050012023600001']

In [32]:
# Make the dictionary

# if do 19780 it works but not for 

# this is the test code that BLS gave
dicto = {
    'SMU19197802023800001': 'Sac, Placer, El D, Yolo'} # testing a different series ID

years = ('2000', '2024')

# Call the func

my_key = '03688c33d1194462aa72e4e97253b1e7'

bls_query(series_dict=dicto, dates=years, api_key=my_key)

,"Sac, Placer, El D, Yolo"
2022-01-01,14.3
2022-02-01,14.4
2022-03-01,14.9
2022-04-01,16.3
2022-05-01,17.0
2022-06-01,17.5
2022-07-01,17.7
2022-08-01,17.7
2022-09-01,17.7
2022-10-01,17.6


In [23]:
# Make the dictionary

dicto = {
    'SMU064090050010500000001': 'Sac, Placer, El D, Yolo',
    'SMU064970050012023600001': 'Yuba, Sutter'} # testing a different series ID

years = ('2000', '2024')

# Call the func

my_key = '03688c33d1194462aa72e4e97253b1e7'

bls_query(series_dict=dicto, dates=years, api_key=my_key)

,"Sac, Placer, El D, Yolo","Yuba, Sutter"


We keep returning empty dataframes when we use our counties. If we use the example query ID 'SMU19197802023800001' which selects Specialty Trade Contractors as the industry and Des Moines-West Des Moines, IA we get a dataframe of length 27. When we ran this query with the original county value (19780) substituted for that of Sacramento county, we got length zero for our frame. It's also important to note that even on our successful pull, we only got data spanning from 2022-2024.

In [34]:
# Using the Same survey but SIC Basis

# Example SeriesID given

# SAS0800002000011 should have data going back to 2000 - 2002

# Supersector code is contained within the industry codes

#ie 20238000 has a 20 in front, which means construction

dicto = {
    'SAS0800002000011': 'Test SeriesID'} # testing a different series ID

years = ('2000', '2024')

# Call the func

my_key = '03688c33d1194462aa72e4e97253b1e7'

bls_query(series_dict=dicto, dates=years, api_key=my_key)

,Test SeriesID


***

## **Trying Peer MSAs**

***

In [96]:
# Reading in Data

peers = pd.read_excel("Area Codes (1).xlsx", 'Common Groups')

peers.head(13)

,SACOG counties,Unnamed: 1
0,states,counties
1,CA,El Dorado
2,CA,Placer
3,CA,Sacramento
4,CA,Sutter
5,CA,Yolo
6,CA,Yuba
7,NaN,NaN
8,Peer MSA's,NaN
9,msa,msa_label


In [97]:
# Subsetting to only keep our peer counties

# Referencing intiial input as well as the excel file

peers = peers.iloc[10:33]

peers.tail()

,SACOG counties,Unnamed: 1
28,41740,"San Diego-Chula Vista-Carlsbad, CA Metro Area"
29,41860,"San Francisco-Oakland-Berkeley, CA Metro Area"
30,41940,"San Jose-Sunnyvale-Santa Clara, CA Metro Area"
31,45300,"Tampa-St. Petersburg-Clearwater, FL Metro Area"
32,49700,"Yuba City, CA Metro Area"


In [98]:
# Attempting to do a pull

# This is using the structure of the test code series id in the documentation: 'SMU19197802023800001'

# Call the func

my_key = '03688c33d1194462aa72e4e97253b1e7'

years = ('2000', '2024')

bls_query(series_dict=dictochat, dates=years, api_key=my_key)

# UPDATE: Accidentally used CA Id for all. 

ValueError: Length of values (27) does not match length of index (0)

### Notes

* Unsure how great this source is, as there is not a ton of records to pull from.

* In my code, need to update it so that each series is returned separately. To further explain, take the example above. The length of values for some counties, are 27 while some are zero.

* Will also need to change the industries. I surmise that some, counties are going to have more data than other i.e big cities. 

In [105]:
# Trying to test bullet point 3

# County is inside the list as well, so should be returning data.

dicto = {
    'SMU06418602023730001': 'San Francisco-Oakland-Berkeley, CA Metro Area'}

In [106]:
# Going to do multiple queries using different industries

# No Results: 2023800, 20236000, 20237300

# Checking the URL link on browser (https://api.bls.gov/publicAPI/v2/timeseries/data/20237300)

# I get "invalid series for series Series 2023700" whenever I do this.
bls_query(series_dict=dicto, dates=years, api_key=my_key)

,"San Francisco-Oakland-Berkeley, CA Metro Area"


In [128]:
# Loading state names as well 

states = pd.read_excel("state names.xlsx")

In [ ]:
# Need the codes to be 00, 01, etc
states.iloc[0:8, 0] = '0' + states.iloc[0:8, 0].astype(str)

C:\Users\jchoy\AppData\Local\Temp\ipykernel_18364\3486777325.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['00' '01' '02' '04' '05' '06' '08' '09']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  states.iloc[0:8, 0] = '0' + states.iloc[0:8, 0].astype(str)


In [161]:
states.head()
peers.head()

,SACOG counties,Unnamed: 1,full_name,State_ID,abbr
10,12420,"Austin-Round Rock-Georgetown, TX Metro Area",Texas,48,TX
11,16740,"Charlotte-Concord-Gastonia, NC-SC Metro Area",North Carolina,37,NC
12,17140,"Cincinnati, OH-KY-IN Metro Area",Ohio,39,OH
13,17460,"Cleveland-Elyria, OH Metro Area",Ohio,39,OH
14,18140,"Columbus, OH Metro Area",Ohio,39,OH


In [143]:
# make a function to extract state abbreviation  

import us

# Gets just abbrev out of the text string in peers

def extract_state(text):
    return text.split(',')[1].strip()

# Converting abbreviation to full name

def abbr_full(abbr):
    return us.states.lookup(abbr).name


# Getting each ID now using full name 

def get_id(full_state_name):
    row = states[states['state_name'] == full_state_name]
    if not row.empty:
        return row.iloc[0]['state_code']
    else:
        return None
    
peers['full_name'] = peers['Unnamed: 1'].apply(extract_state)

peers['abbr'] = peers['full_name'].str[:2]

peers['full_name'] = peers['abbr'].apply(abbr_full)

peers['State_ID'] = peers['full_name'].apply(get_id)

In [155]:
# Changing state id to be string

peers['State_ID'] = peers['State_ID'].astype(str)

In [156]:
# Create a function to make all of these counties into a dictionary

def dict_maker(df):

    # Formula for Series ID = Prefix + SA + State + Area + Industry + DType
    pre = "SMU"
    
    supersector = "20238000" # We can make this be chosen

    data_type = '01'

    # Making set of keys and vals for future dict
    
    keys = []

    # Have to initialize as we can't use 0 in for loop

    # defo a better way

    first_key = pre + '48' + df.iloc[0, 0] + supersector + data_type

    keys.append(first_key)

    # Now empty list for values in the future dict

    vals = []

    first_val = df.iloc[0,1]
    
    vals.append(first_val)

    for i in range(len(df)):

        # Getting each code
        area_code = df.iloc[i, 0]

        state = peers.iloc[i, 3]

        # Making each SeriesID
        series_id = pre + state + area_code + supersector + data_type

        # Adding to the keylist for future dictionary
        keys.append(series_id)

        val = df.iloc[i, 1]

        vals.append(val)

    result = {k: v for k, v in zip(keys, vals)}

    return result
        
dictochat = dict_maker(df = peers)

In [157]:
dictochat

{'SMU48124202023800001': 'Austin-Round Rock-Georgetown, TX Metro Area',
 'SMU37167402023800001': 'Charlotte-Concord-Gastonia, NC-SC Metro Area',
 'SMU39171402023800001': 'Cincinnati, OH-KY-IN Metro Area',
 'SMU39174602023800001': 'Cleveland-Elyria, OH Metro Area',
 'SMU39181402023800001': 'Columbus, OH Metro Area',
 'SMU26198202023800001': 'Detroit-Warren-Dearborn, MI Metro Area',
 'SMU18269002023800001': 'Indianapolis-Carmel-Anderson, IN Metro Area',
 'SMU29281402023800001': 'Kansas City, MO-KS Metro Area',
 'SMU12331002023800001': 'Miami-Fort Lauderdale-Pompano Beach, FL Metro Area',
 'SMU12367402023800001': 'Orlando-Kissimmee-Sanford, FL Metro Area',
 'SMU04380602023800001': 'Phoenix-Mesa-Chandler, AZ Metro Area',
 'SMU42383002023800001': 'Pittsburgh, PA Metro Area',
 'SMU41389002023800001': 'Portland-Vancouver-Hillsboro, OR-WA Metro Area',
 'SMU06401402023800001': 'Riverside-San Bernardino-Ontario, CA Metro Area',
 'SMU06409002023800001': 'Sacramento-Roseville-Folsom, CA Metro Area

In [158]:
# Running with fixed code

bls_query(series_dict=dictochat, dates=years, api_key=my_key)

ValueError: Length of values (0) does not match length of index (27)

Same issue as before, but this should be an easy fix. Again, 27 is the max length we've observed. Farthest data I've seen this survey go back is 2014, is this enough?

In [165]:
#SMU06409009000000001

sac_dict = {
    'SMU06409009000000001': 'Sac, Placer, El D, Yolo'} # testing a different series ID

years = ('2014', '2024')

# Call the func

my_key = '03688c33d1194462aa72e4e97253b1e7'

bls_query(series_dict=sac_dict, dates=years, api_key=my_key)

,"Sac, Placer, El D, Yolo"
2014-01-01,225.3
2014-02-01,227.5
2014-03-01,228.6
2014-04-01,230.9
2014-05-01,231.5
...,...
2023-08-01,252.3
2023-09-01,255.1
2023-10-01,258.9
2023-11-01,262.0


In [166]:
# Let's update bls_query() so that we can make it so that each series is uniform and has 120 rows for all

def bls_query_update(series_dict, dates, api_key):

    url = 'https://api.bls.gov/publicAPI/v2/timeseries/data/'

    key = '?registrationkey={}'.format(api_key)

    # Specify json as content type to return
    headers = {'Content-type': 'application/json'}

    # Submit the list of series as data
    data = json.dumps({
        "seriesid": list(series_dict.keys()),
        "startyear": dates[0],
        "endyear": dates[1]})

    # Post request for the data
    p = requests.post(
        '{}{}'.format(url, key),
        headers=headers,
        data=data).json()['Results']['series']
    
    # Date index from first series
    date_list = [f"{i['year']}-{i['period'][1:]}-01" for i in p[0]['data']]

    global df

    # Empty dataframe to fill with values
    df = pd.DataFrame()

    # Build a pandas series from the API results, p

    # We want to make all industries consistent, meaning all counties have 120 rows
    for s in p:
        county_name = series_dict[s['seriesID']]
        county_data = {f"{i['year']}-{i['period'][1:]}-01": float(i['value']) if 'value' in i else 0 for i in s['data']}

        df[county_name] = pd.Series(county_data)

    df.index = pd.to_datetime(date_list)

    return(df)

bls_query_update(series_dict=dictochat, dates = (2000, 2024), api_key = my_key)

,"Austin-Round Rock-Georgetown, TX Metro Area","Charlotte-Concord-Gastonia, NC-SC Metro Area","Cincinnati, OH-KY-IN Metro Area","Cleveland-Elyria, OH Metro Area","Columbus, OH Metro Area","Detroit-Warren-Dearborn, MI Metro Area","Indianapolis-Carmel-Anderson, IN Metro Area","Kansas City, MO-KS Metro Area","Miami-Fort Lauderdale-Pompano Beach, FL Metro Area","Orlando-Kissimmee-Sanford, FL Metro Area",...,"Riverside-San Bernardino-Ontario, CA Metro Area","Sacramento-Roseville-Folsom, CA Metro Area","St. Louis, MO-IL Metro Area","Salt Lake City, UT Metro Area","San Antonio-New Braunfels, TX Metro Area","San Diego-Chula Vista-Carlsbad, CA Metro Area","San Francisco-Oakland-Berkeley, CA Metro Area","San Jose-Sunnyvale-Santa Clara, CA Metro Area","Tampa-St. Petersburg-Clearwater, FL Metro Area","Yuba City, CA Metro Area"
2009-12-01,24.0,31.9,24.1,20.9,17.7,34.8,25.0,NaN,61.1,32.5,...,42.8,25.9,NaN,21.4,27.2,38.2,NaN,23.2,38.2,NaN
2009-11-01,24.1,32.1,24.8,22.1,18.5,36.7,25.9,NaN,61.8,32.8,...,44.5,27.2,NaN,22.3,27.4,38.5,NaN,23.6,38.7,NaN
2009-10-01,24.2,32.4,25.3,22.9,18.7,37.9,26.4,NaN,63.1,33.3,...,45.1,27.7,NaN,22.8,27.7,38.9,NaN,23.6,39.5,NaN
2009-09-01,24.6,33.2,25.5,22.8,19.2,37.5,26.8,NaN,65.0,33.6,...,46.4,27.9,NaN,22.8,28.4,39.3,NaN,23.9,40.6,NaN
2009-08-01,24.7,33.7,25.9,23.1,19.8,38.2,27.3,NaN,65.5,34.4,...,47.7,28.5,NaN,22.8,28.8,40.3,NaN,24.2,41.1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2000-05-01,25.9,37.6,33.5,31.9,28.5,69.4,NaN,NaN,NaN,34.3,...,55.8,35.4,NaN,23.7,NaN,45.0,NaN,36.6,46.6,NaN
2000-04-01,25.5,37.1,32.7,30.4,27.7,66.9,NaN,NaN,NaN,34.5,...,54.0,34.2,NaN,23.3,NaN,44.3,NaN,35.8,46.2,NaN
2000-03-01,25.4,36.8,32.3,29.4,26.2,62.2,NaN,NaN,NaN,34.5,...,52.5,32.8,NaN,22.9,NaN,44.0,NaN,35.1,45.8,NaN
2000-02-01,24.4,35.8,31.1,27.7,25.1,59.6,NaN,NaN,NaN,34.3,...,51.9,32.0,NaN,22.8,NaN,44.1,NaN,34.1,45.0,NaN


***

## **Pulling All Subsectors With All Peers**

***

In [167]:
peers

,SACOG counties,Unnamed: 1,full_name,State_ID,abbr
10,12420,"Austin-Round Rock-Georgetown, TX Metro Area",Texas,48,TX
11,16740,"Charlotte-Concord-Gastonia, NC-SC Metro Area",North Carolina,37,NC
12,17140,"Cincinnati, OH-KY-IN Metro Area",Ohio,39,OH
13,17460,"Cleveland-Elyria, OH Metro Area",Ohio,39,OH
14,18140,"Columbus, OH Metro Area",Ohio,39,OH
15,19820,"Detroit-Warren-Dearborn, MI Metro Area",Michigan,26,MI
16,26900,"Indianapolis-Carmel-Anderson, IN Metro Area",Indiana,18,IN
17,28140,"Kansas City, MO-KS Metro Area",Missouri,29,MO
18,33100,"Miami-Fort Lauderdale-Pompano Beach, FL Metro ...",Florida,12,FL
19,36740,"Orlando-Kissimmee-Sanford, FL Metro Area",Florida,12,FL


In [171]:
# Create a function to make all of these counties into a dictionary

def dict_maker(df, sector):

    # Formula for Series ID = Prefix + SA + State + Area + Industry + DType
    pre = "SMU"
    
    data_type = '01'

    # Making set of keys and vals for future dict
    
    keys = []

    # Have to initialize as we can't use 0 in for loop

    # defo a better way

    first_key = pre + '48' + df.iloc[0, 0] + df.iloc[0,3] + data_type

    keys.append(first_key)

    # Now empty list for values in the future dict

    vals = []

    first_val = df.iloc[0,1]
    
    vals.append(first_val)

    for i in range(len(df)):

        # Getting each code
        area_code = df.iloc[i, 0]

        state = peers.iloc[i, 3]

        # Making each SeriesID
        series_id = pre + state + area_code + sector + data_type

        # Adding to the keylist for future dictionary
        keys.append(series_id)

        val = df.iloc[i, 1]

        vals.append(val)

    result = {k: v for k, v in zip(keys, vals)}

    return result
        
dictochat = dict_maker(df = peers)

TypeError: dict_maker() missing 1 required positional argument: 'sector'

In [ ]:
# Need to update bls_query() so that it doesn't only do the first 10 years

# Let's update bls_query() so that we can make it so that each series is uniform and has 120 rows for all

def bls_query_update(series_dict, dates, api_key):

    url = 'https://api.bls.gov/publicAPI/v2/timeseries/data/'

    key = '?registrationkey={}'.format(api_key)

    start_year = dates[0]

    end_year = dates[1]

    # Specify json as content type to return
    headers = {'Content-type': 'application/json'}

    # Initialize empty df for assignment later

    df = pd.DataFrame()

    # Loop through the specified range of years in 10-year intervals
    for year_range_start in range(start_year, end_year, 10):
        year_range_end = min(year_range_start + 9, end_year)  # Ensure the end year is within the specified range

        # Specify the date range for the current iteration
        dates = (year_range_start, year_range_end)

        # Submit the request for the current date range
        data = json.dumps({
            "seriesid": list(series_dict.keys()),
            "startyear": dates[0],
            "endyear": dates[1]})
        response = requests.post('{}{}'.format(url, key), headers=headers, data=data).json()

        # Extract data from the response and concatenate to the DataFrame
        if 'Results' in response and 'series' in response['Results']:
            p = response['Results']['series']
            date_list = [f"{i['year']}-{i['period'][1:]}-01" for i in p[0]['data']]
            temp_df = pd.DataFrame(index=pd.to_datetime(date_list))

            for s in p:
                county_name = series_dict[s['seriesID']]
                county_data = {f"{i['year']}-{i['period'][1:]}-01": float(i['value']) if 'value' in i else None for i in s['data']}
                temp_df[county_name] = pd.Series(county_data)

            df = pd.concat([df, temp_df], axis=0)

    return df


# Now, it should return a single dataframe for a single subsector.

# This frame should contain all of the counties, with dates from 2000-2024.

# Additionally, if a county doesn't have data for a specific month, that cell should be empty

bls_query_update(series_dict=dictochat, dates = (2000, 2024), api_key = my_key)

In [209]:
# Now let's run the final function

# Should also try and make it so that you can choose what counties you want.

# This will be done when we make the function in tandem with excel

def full_bls(sector_list, df, key):
    
    # Initialize an empty list so we can iterate over multiple dictionaries
    sector_chamber = []

    # Initialize empty list for each dataframe we will end up making
    df_chamber = []

    # Loop over each sector we want to test
    for i in sector_list:
        sector_chamber.append(dict_maker(df, i))

    # Now with the sector_holders list containing each set of series we want, we can run our query function iteratively
    
    # Iteratively make each dataframe
    for sector_dict in sector_chamber:
        df_chamber.append(bls_query_update(sector_dict, dates = (2000, 2024), api_key = key))

    return(df_chamber)
# Perhaps we can multiprocess this. For loops very bad for efficiency, and this is going to be a p lengthy process. 


In [ ]:
import multiprocessing
import time

# This is a multiprocess function for improved efficiency.
def full_bls(sector_list, df, key):

    # Empty list to store results
    results = []

    # Define a function to process each sector, use time for rate limit
    def process_sector(sector):
        time.sleep(10)
        return bls_query_update(dict_maker(df, sector), dates=(2000, 2024), api_key=key)

    # Create a pool of worker processes
    with multiprocessing.Pool(processes=multiprocessing.cpu_count()) as pool:
        
        # Map the sectors to worker processes
        results = pool.map(process_sector, sector_list)

    return results

The API returns up to 10 years of data for up to 25 time series. We will have to separate the queries in half, and then stack the frames on top of each other. 

Also, we can actually pull all the way back to 1990. So maybe we have to do this process three times. 

In [190]:
# Testing Agg = Total Nonfarm - Total Priv

sectors = ['00000000', '05000000', '08000000']
dfs = full_bls(sectors, peers, my_key)

In [192]:
# Trying to find agriculture
trade = ['41000000']
trade_df = full_bls(trade, peers, my_key)

In [194]:
trade_df[0]

,"Austin-Round Rock-Georgetown, TX Metro Area","Charlotte-Concord-Gastonia, NC-SC Metro Area","Cincinnati, OH-KY-IN Metro Area","Cleveland-Elyria, OH Metro Area","Columbus, OH Metro Area","Detroit-Warren-Dearborn, MI Metro Area","Indianapolis-Carmel-Anderson, IN Metro Area","Kansas City, MO-KS Metro Area","Miami-Fort Lauderdale-Pompano Beach, FL Metro Area","Orlando-Kissimmee-Sanford, FL Metro Area",...,"Riverside-San Bernardino-Ontario, CA Metro Area","Sacramento-Roseville-Folsom, CA Metro Area","St. Louis, MO-IL Metro Area","Salt Lake City, UT Metro Area","San Antonio-New Braunfels, TX Metro Area","San Diego-Chula Vista-Carlsbad, CA Metro Area","San Francisco-Oakland-Berkeley, CA Metro Area","San Jose-Sunnyvale-Santa Clara, CA Metro Area","Tampa-St. Petersburg-Clearwater, FL Metro Area","Yuba City, CA Metro Area"
2009-12-01,30.9,48.4,55.1,46.5,36.4,77.5,45.3,45.6,134.5,38.8,...,48.1,22.7,58.5,28.2,28.6,41.5,64.9,34.3,46.1,1.0
2009-11-01,30.8,48.5,55.0,46.8,36.5,77.3,45.3,45.8,133.7,38.8,...,48.0,23.1,58.1,28.1,28.6,41.6,65.1,34.4,45.9,1.1
2009-10-01,30.7,48.8,54.9,47.0,36.4,77.5,45.4,46.0,132.7,39.0,...,48.0,23.2,58.4,28.2,28.6,41.6,65.3,34.4,46.0,1.1
2009-09-01,30.8,48.6,54.7,47.2,36.6,77.2,45.2,46.0,132.0,39.2,...,47.8,23.2,58.1,28.2,28.4,41.2,65.0,34.2,45.8,1.0
2009-08-01,31.1,49.1,55.2,47.7,37.1,77.6,45.7,46.5,132.6,39.4,...,48.0,23.4,58.5,28.4,28.5,41.5,65.7,34.5,46.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2000-05-01,29.2,53.4,59.2,57.4,39.3,103.7,49.5,48.8,124.5,40.3,...,38.6,25.2,57.8,27.1,26.5,40.3,84.8,42.0,55.5,0.9
2000-04-01,28.8,53.2,59.0,57.1,39.1,102.9,49.2,48.8,123.8,40.0,...,38.3,25.2,57.7,26.9,26.4,39.9,84.4,41.8,55.0,0.9
2000-03-01,28.6,52.8,58.7,56.9,38.7,101.7,49.3,47.7,121.7,38.4,...,36.1,24.8,57.3,26.9,26.5,39.5,84.1,42.0,54.8,0.9
2000-02-01,28.5,52.3,58.6,56.8,38.6,101.5,48.9,47.6,120.8,38.1,...,35.9,24.6,57.0,26.7,26.3,39.7,83.0,41.9,54.1,0.9


In [200]:
# Let's test this first function

sectors = ['00000000', '10000000', '20000000']
dfs = full_bls(sectors, peers, my_key)

In [208]:
print(dfs[0].shape)
print(dfs[1].shape)
print(dfs[2].shape)

(291, 23)
(231, 23)
(231, 23)
